# PED 6.2 · NOTEBOOK 
## 1. Análisis básico de datos climáticos con ERA5

En esta práctica se trabajará con datos del reanálisis ERA5 descargados por el/la propio/a estudiante desde el portal Copernicus Climate Data Store (CDS).

La tarea es doble:

1. representar una serie temporal diaria de precipitación para un punto próximo a Madrid;
2. elaborar un mapa de precipitación sobre la Comunidad de Madrid a partir de datos en rejilla.

La práctica permite familiarizarse con un flujo de trabajo muy habitual en análisis climático:

- localización y descarga de datos;
- carga del archivo en un entorno de análisis;
- inspección de variables y dimensiones;
- transformación básica de unidades y escalas temporales;
- representación gráfica e interpretación.



## 2. Descarga previa de los datos

Antes de ejecutar este notebook, descargue desde esta página de Copernicus los archivos necesarios:
https://cds.climate.copernicus.eu/datasets

### 2.1 Archivo 1. Serie temporal puntual
Dataset recomendado:
**ERA5 hourly time-series data on single levels from 1940 to present**. 

Configuración orientativa:
- Variable: `total precipitation`
- Localización: Madrid (por ejemplo, latitud 40.42; longitud -3.70)
- Periodo: junio-agosto de 2019
- Formato: CSV

### 2.2 Archivo 2. Campo espacial para mapa
Dataset recomendado:
**ERA5 hourly data on single levels from 1940 to present**.

Configuración orientativa:
- Variable: `total precipitation`
- Área: entorno de la Comunidad de Madrid (North: 42; South: 39; West: -5; East: -2)
- Fecha: seleccione todos los datos del verano de 2019 (1 de junio al 31 de agosto)
- Hora: Descargue todas las horas del día
- Formato: NetCDF

#### Sobre unidades

En ERA5, la precipitación total (`total precipitation`, `tp`) se expresa en **metros de agua**. Para expresarla en milímetros, se multiplica por 1000. 

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

## 3. Serie temporal de precipitación en un punto

En este primer apartado trabajaremos con el archivo CSV descargado para un punto próximo a Madrid.

En esta parte queremos obtener una **serie diaria de precipitación** a partir de datos horarios.

### Uso del archivo en el entorno de trabajo

Para poder utilizar el archivo en este notebook, es necesario cargarlo previamente en el entorno de trabajo de Binder.

Para ello:

- localice el panel de archivos en la parte izquierda de la pantalla (recuerde que tiene que estar en el entorno JupyterLab, ver archivo de la PED);
- arrastre el archivo CSV descargado desde su ordenador a ese panel;
- compruebe que el archivo aparece en la lista de archivos disponibles.

Una vez cargado, podrá utilizar su nombre directamente en el código.

### Acción

Escriba en la variable `csv_file` el **nombre exacto del archivo** (incluyendo la extensión `.csv`) tal como aparece en el panel de la izquierda. Ejecute la celda y observe la tabla cargada.

In [ ]:
csv_file = "AQUI_ESCRIBA_EL_NOMBRE_DE_SU_ARCHIVO.csv"

df = pd.read_csv(csv_file)
df.head()

### Pregunta 1
En base a la tabla responda brevemente:

- ¿qué columnas aparecen?
- ¿qué columna contiene la fecha y hora?
- ¿qué columna contiene la precipitación?

In [ ]:
df.columns

## 4. Conversión de la fecha y hora

Convertimos la columna temporal a un formato de fecha y hora reconocido por la librería Phyton 'pandas', que permite trabajar correctamente con series temporales.

Después, se crea una nueva columna que contenga solo la fecha, sin la hora.

In [ ]:
# Sustituya el nombre de la columna temporal si fuera necesario
df["valid_time"] = pd.to_datetime(df["valid_time"])

# Cree una columna solo con la fecha
df["date"] = df["valid_time"].dt.date

df.head()

### Unidades de precipitación

En ERA5, la variable `total precipitation` (`tp`) se expresa en metros de agua. Para pasar a milímetros, se multiplica por 1000. Para ello se crea una nueva columna con la precipitación en milímetros.

In [ ]:
# Sustituya 'tp' por el nombre real de la columna si fuera distinto
df["tp_mm"] = df["tp"] * 1000

df[["valid_time", "tp", "tp_mm"]].head()

## 5. De datos horarios a precipitación diaria

A diferencia de la temperatura media, la precipitación diaria no se obtiene promediando los valores horarios, sino **sumándolos**. Para ellos se agrupan los datos por día y se calcula la precipitación total diaria.

In [ ]:
daily_precip = (
    df.groupby("date", as_index=False)["tp_mm"]
      .sum()
      .rename(columns={"tp_mm": "precip_mm_day"})
)

daily_precip.head()

## 6. Representación de la serie temporal

Se representa la serie diaria obtenida. Asigne nombres apropiados para substituir a las mayúsculas, para los nombres de los ejes y el título de la gráfica.



In [ ]:
plt.figure(figsize=(10,4))
plt.plot(daily_precip["date"], daily_precip["precip_mm_day"])
plt.xlabel("EJE X")
plt.ylabel("EJE Y (UNIDADES)")
plt.title("TITULO")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Pregunta 2
Adjunte la figura obtenida, describiendo brevemente la precipitación en el periodo analizado (abundancia/escasez, patrón...).


## 7. Mapa de precipitación sobre la Comunidad de Madrid

En este segundo apartado trabajaremos con un archivo NetCDF descargado desde ERA5 en rejilla.

El objetivo es generar un mapa de precipitación para la región de Madrid.
###Acción
Siguiendo las mismas instrucciones que con el archivo anterior, cargue en el entorno de trabajo el archivo de extensión NC.

In [ ]:
nc_file = "AQUI_ESCRIBA_EL_NOMBRE_DE_SU_ARCHIVO.nc"
ds = xr.open_dataset(nc_file)
ds

### Pregunta 3
A partir de la salida del dataset, indique:

- nombre de la variable principal de precipitación;
- dimensiones disponibles;
- coordenadas espaciales incluidas.

In [ ]:
list(ds.data_vars)

Se convierten unidades (de m a mm). 

In [ ]:
tp = ds["tp"] * 1000  # conversión de m a mm

tp

## 8. Construcción de un mapa diario

Como el archivo contiene varias horas del mismo día, se obtiene la precipitación diaria sumando a lo largo de la dimensión temporal.

In [ ]:
# Esta celda asume que el archivo contiene varias horas del mismo día
tp_summer = tp.sum(dim="valid_time")

tp_summer

Se puede recortar la 'ventana' en la que queremos mostrar la información territorial en el mapa (en una primera 'ejecución' del código use estos valores; modifíquelos una vez visualizado el mapa con los límites administrativos para ajustar mejor sus dimensiones).

In [ ]:
tp_madrid = tp_summer.sel(
    latitude=slice(42, 39),
    longitude=slice(-5.0, -2)
)

tp_madrid

### Representación del mapa

Se representa el mapa con los datos procesados. Asigne nombres apropiados para substituir a las mayúsculas, para los nombres de los ejes y el título de la gráfica.


In [ ]:

plt.figure(figsize=(7,6))
tp_madrid.plot(cmap="Blues",cbar_kwargs={"label": "Precipitación acumulada (mm)"})
plt.title("TÍTULO GRÁFICA")
plt.xlabel("EJE X")
plt.ylabel("EJE Y")
plt.tight_layout()
plt.show()

### Pregunta 4
Adjunte el mapa obtenido, describiendo brevemente el patrón espacial y sus limitaciones.

## 9. Representación de información geográfica (límites administrativos)

Para añadir información espacial adicional al mapa (en este caso, el límite de la Comunidad de Madrid), se pueden utilizar archivos en formato **GeoJSON**.

Un archivo GeoJSON contiene información geográfica (puntos, líneas o polígonos) junto con atributos asociados, y es un formato ampliamente utilizado en análisis geoespacial por su simplicidad y compatibilidad con distintas herramientas.

En esta práctica, se proporciona un archivo con los límites de las comunidades autónomas de España (`spain-communities.geojson`), que está disponible en el repositorio del proyecto. Estos archivos pueden encontrarse fácilmente en repositorios públicos de datos geográficos (por ejemplo, en plataformas abiertas como GitHub o portales de datos oficiales) y descargarse directamente en formato GeoJSON.

A partir de este archivo, se selecciona únicamente la Comunidad de Madrid y se superpone sobre el mapa de precipitación.

Este procedimiento ilustra un flujo habitual en análisis climático y ambiental:

- utilización de datos en rejilla (como ERA5) para representar variables climáticas;
- incorporación de capas geográficas externas (límites administrativos, regiones, etc.);
- combinación de ambas fuentes para facilitar la interpretación de los resultados.

Este mismo enfoque puede utilizarse con otros conjuntos de datos geográficos (por ejemplo, provincias, cuencas hidrográficas o áreas protegidas), siempre que estén disponibles en formatos estándar como GeoJSON o shapefile.

Se representa ahora el mapa anterior junto con los límites administrativos. Asigne nombres apropiados para substituir a las mayúsculas, para los nombres de los ejes y el título de la gráfica.


In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd

spain = gpd.read_file("spain-communities.geojson")
madrid = spain[spain["name"] == "Madrid"].copy()
madrid = madrid.to_crs("EPSG:4326")

fig, ax = plt.subplots(figsize=(8, 7))

tp_madrid.plot(
    ax=ax,
    cmap="Blues",
    cbar_kwargs={"label": "Precipitación acumulada (mm)"}
)

madrid.boundary.plot(ax=ax, edgecolor="red", linewidth=2)

ax.scatter(-3.70, 40.42, color="yellow", s=40)   # Madrid

ax.set_title("TITULO GRÁFICA")
ax.set_xlabel("EJE X")
ax.set_ylabel("EJE Y")

plt.show()

### Pregunta 5. 

Adjunte el mapa obtenido y añada un breve comentario sobre la escala espacial de los datos de reanálisis. 

## Pregunta 6. 

Como ampliación haga una representación de una serie temporal de datos y de un mapa para otras elecciones de variable, espacio (otra localización) y período. No es necesario incluir mapa de límites administrativos u otros datos no climáticos. Adjunte el resultado, especificando qué representa y el origen y tipología de los datos utilizados.



Acceda al visor de datos climáticos de AdapteCCa
https://escenarios.adaptecca.es/
para buscar en este portal datos equivalentes a los tratados en esta PED.  Explore visualmente resultados equivalentes a los obtenidos en esta PED. 
    
### Pregunta 7

Haga una comparación (ventajas e inconvenientes) entre las tres formas posibles de acceder y manipular datos: mediante archivos csv directamente en excel, en Phyton o a través de plataformas como AdapteCCa.
